# 全天分钟信息进入既有 SOXX/SOXL/cash 神经分配器

执行前冻结配置见 `plan.json`。保留 2011 年以来的日频训练历史；分钟数据缺失保持 NaN 和可用标记。用同一网络、损失、持仓约束和费用比较 base、availability_control、core_path、peer_context、core_peer，保留三个 seed。模型选择只使用 2020–2022 的成熟开发期收益，2023–2026 是复用诊断，不构成独立前向。

新特征使用不复权的同一交易时段路径，避免把未来公开复权系数写进历史分钟收益。

In [1]:
from pathlib import Path
import sys,json,os
root=Path.cwd().resolve()
while not (root/"AGENTS.md").exists() and root!=root.parent:root=root.parent
sys.path.insert(0,str(root))
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT","1")
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.workflow import R
card=root/"research/minute_information_pit_allocator_20261012"
plan=json.loads((card/"plan.json").read_text())
plan["maximum_model_fits"],plan["maximum_native_cases"]

(54, 42)

## 原生 Qlib 数据与因果特征

从 NAS 的固定源恢复 raw NONE，在独立研究 provider 中读取，保护运行中的分钟 store。日频和辅助数据也通过 Qlib 读取，冻结进原生 Recorder。

In [2]:
from src.workflow.minute_pit_allocator import prepare_minute_allocator_inputs
inputs=prepare_minute_allocator_inputs(root=root,card=card,plan=plan)
inputs["audit"]

{'source_observations': 2184168,
 'minute_source_id': '18f1fb3f17907932ddd13881aea9199369181a92c9b6c77395d0b255100c68c3',
 'minute_price_basis': 'raw NONE within-session',
 'minute_sessions': 2008,
 'valid_core_sessions': 2007,
 'valid_peer_sessions': 2007,
 'full_daily_rows': 3966,
 'daily_first_signal': '2011-01-03',
 'daily_last_signal': '2026-10-09',
 'original_features': ['soxl_ret3',
  'soxl_ret5',
  'soxl_ret10',
  'soxl_ret20',
  'soxl_ret60',
  'soxx_ret3',
  'soxx_ret5',
  'soxx_ret10',
  'soxx_ret20',
  'soxx_ret60',
  'relative_momentum20',
  'soxx_trend20',
  'soxx_trend60',
  'trend_efficiency_20',
  'levered_decay_20',
  'soxx_lead_10',
  'range_pos_252',
  'cross_breadth_pos_20',
  'cross_breadth_pos_60',
  'semis_vs_cross_20',
  'semis_vs_cross_60',
  'tech_breadth_20',
  'tech_breadth_60',
  'soxx_vol20',
  'soxx_drawdown60',
  'dist_high_252',
  'vix',
  'vix_chg5',
  'vix_z_60',
  'rv20',
  'rv60',
  'rv20_expansion',
  'tracking_error_5_z20',
  'tracking_error_5_pc

## 同结构训练、固定特征对照与原生账户

训练标准化只使用已成熟训练样本；未知分钟通道在模型内部中性编码，原始观察不填造。新输入权重初始为零，保留相同原网络初始化。EMA/HOLD 判定与费用在每个账户独立结算。

In [3]:
from src.workflow.minute_pit_allocator import run_minute_allocator_campaign
summary=run_minute_allocator_campaign(root=root,card=card,plan=plan,inputs=inputs)
{k:summary[k] for k in ["decision","selected_family","model_fits","native_cases","incremental_development_gate","paired_seed_gate","benchmark_and_risk_gate","compact_metrics"]}


fit 1/54 base_2020_71 n=2244
fit 2/54 base_2020_172 n=2244
fit 3/54 base_2020_273 n=2244
fit 4/54 availability_control_2020_71 n=2244
fit 5/54 availability_control_2020_172 n=2244
fit 6/54 availability_control_2020_273 n=2244
fit 7/54 core_path_2020_71 n=2244
fit 8/54 core_path_2020_172 n=2244
fit 9/54 core_path_2020_273 n=2244
fit 10/54 peer_context_2020_71 n=2244
fit 11/54 peer_context_2020_172 n=2244
fit 12/54 peer_context_2020_273 n=2244
fit 13/54 core_peer_2020_71 n=2244
fit 14/54 core_peer_2020_172 n=2244
fit 15/54 core_peer_2020_273 n=2244
fit 16/54 base_2021_71 n=2497
fit 17/54 base_2021_172 n=2497
fit 18/54 base_2021_273 n=2497
fit 19/54 availability_control_2021_71 n=2497
fit 20/54 availability_control_2021_172 n=2497
fit 21/54 availability_control_2021_273 n=2497
fit 22/54 core_path_2021_71 n=2497
fit 23/54 core_path_2021_172 n=2497
fit 24/54 core_path_2021_273 n=2497
fit 25/54 peer_context_2021_71 n=2497
fit 26/54 peer_context_2021_172 n=2497
fit 27/54 peer_context_2021_273

{'benchmark_and_risk_gate': False,
 'compact_metrics': [{'cagr_pct': 17.025029068345177,
                      'cost_bps': 10,
                      'cvar5_pct': -6.216418984352099,
                      'max_drawdown_pct': -50.79311874943225,
                      'return_pct': 60.064262566988965,
                      'trade_days': 495,
                      'turnover': 29.079351426425422,
                      'wealth_ratio_vs_availability_control': 1.1067960229699536,
                      'wealth_ratio_vs_base': 1.0833931171680828,
                      'wealth_ratio_vs_frozen_current': 0.9694547934422076,
                      'wealth_ratio_vs_soxx': 1.1425205898659159,
                      'window': 'development'},
                     {'cagr_pct': 15.331259803737108,
                      'cost_bps': 25,
                      'cvar5_pct': -6.22226638929707,
                      'max_drawdown_pct': -51.46593995958031,
                      'return_pct': 53.23199816362232,
    

## 最终中文结论

原生实验已执行；最终交付补充净收益归因、软件检查与 NAS 读回。未重新训练或回放。


In [4]:
from IPython.display import Markdown,display
display(Markdown((card/"conclusion.md").read_text()))

# 中文结论

- 执行：NAS raw NONE 的全天分钟信息直接进入现役同结构的 PIT 辅助神经分配器；原有 39 项输入保留。固定五组输入、三个 seed、54 次拟合、42 个原生 Qlib 账户。
- 数据：分钟源 2,184,168 条观察；3966 个日频信号保留 2011 年以来完整历史。有效核心/上下文交易日分别 2007/2007；缺分钟通道为 NaN 和可用标记，仅在模型内部中性编码。源异常压力日没有从账户日历删去。
- 特征：核心 12 项、上下文 8 项，使用同一交易时段的不复权价格比率，无 VWAP 特征和未来复权系数。分组 IC 仅为诊断，未按单项历史相关性挑选特征。
- 训练：固定末轮 checkpoint、300 epochs；20 个原生交易日 purge，主标签真实 H5 成熟时点严格早于 cutoff；辅助标签从每个 cutoff 之前的 Qlib 原始观察重建。共同原网络初始化，新输入初始权重为零。
- 窗口：2020–2022 成熟开发期选择 core_path；2023–2026 为复用诊断，账户最终退出日 2026-10-09。最新 2026-10-09 信号保留未成熟标签，可生成预测，未向 broker 发出目标。
- 费用：10/25bp 各自拥有现金、持仓、EMA/HOLD 决策与实际费用；每个路径经原生 Qlib Position/Account 读回核对。冻结父模型只是同输入的神经网络参考，不代表现役 Stage1 动作层路径。
- 结果：增量开发门=False，配对 seed 门=True，开发年份门=True，SOXX 收益与风险门=False。各费用收益、回撤、CVaR、年度差异见 backtest_benchmark_comparison.csv、regime_metrics.csv；家庭校正区块区间见 uncertainty_summary.csv。
- 晋升：not_promoted_minute_mlp_gates_failed；独立合格前向=0，objective_achieved=false，现役模型/动作层未切换。历史提升与稳定前向超额是不同结论。
- 风险：历史首次收到行情和完整时点资格未认证；每日开盘历史执行不声称真实分钟成交；原有日频来源、PIT 辅助样本与现金零利息口径继承原模型，不以本研究补发来源资格。
- 下一步：结束本次固定特征路线，保留全部失败与对照；只有具备新信息或独立前向证据的新授权研究才继续，不追加参数搜索。
- 原生 Recorder：68e31a0a0fad44e7ba1d1f7d856c7a0d；行情与大研究对象留在 NAS，Git 只交代码、notebook、瘦卡与引用。

## 结果归因

- 10bp 复用诊断：核心分钟模型净收益 385.60%，同条件重训基线 384.46%，SOXX 399.09%；核心最大回撤 -41.23%，基线 -47.27%，SOXX -42.79%。
- 25bp 复用诊断：核心分钟模型净收益 357.66%，同条件重训基线 364.12%，SOXX 398.34%；核心最大回撤 -42.18%，基线 -47.90%，SOXX -42.79%。
- 核心路径改善了重训基线的回撤，但 25bp 下新增收益为负，校正区间跨零；2023、2024、2026 年相对重训基线为负，2025 年为正。上下文及合并组开发期落后。
- 相对旧单模型的收益提升不能归因于分钟特征：没有新增分钟数值的重训基线和可用性对照同样显著改善；新训练协议、三个 seed 的平均、来源更新与严格成熟规则也发生了变化。固定诊断选择不因外层对照排序改变。
- 新分钟数据实际进入 53/49/61 输入的神经网络；原有 39 输入基线保留，所有模型和真实原生账户留在 Recorder。2020 年严格 cutoff 下 PIT 辅助行数为 0，2021/2022/outer 为 6,578/13,007/20,741；不补造缺失辅助样本。
- 验证：25 项聚焦检查通过；另有一次不计研究评分的真实数据两轮集成检查。原 4 项旧生产检查未重跑、未改变其失败状态。
